In [ ]:
import numpy as np # linear algebra
import polars  as pl # data processing, CSV file I/O (e.g. pl.scan_csv)

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(rc={"figure.figsize": (20, 10)})

import os
import warnings
warnings.simplefilter("ignore")

print ("what is left?")

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


for dirname, _, filenames in os.walk('/kaggle/working'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


In [ ]:
kaggle_path = '/kaggle/input/playground-series-s5e1'

train_df = pl.scan_csv(f'{kaggle_path}/train.csv').collect()
test_df = pl.scan_csv(f'{kaggle_path}/test.csv').collect()
sample_df = pl.scan_csv(f'{kaggle_path}/sample_submission.csv').collect()
gdp_per_capita = pl.scan_csv('/kaggle/input/gdp-per-capita/gdp-per-capita-maddison.csv').collect()

display (train_df.collect_schema())

display (train_df.head(5))

# EDA

In [ ]:
create_EDA = False 

In [ ]:
print ("unique values for country, store, product")
print (train_df.get_column ('country').unique().to_list())
print (train_df.get_column ('store').unique().to_list())
print (train_df.get_column ('product').unique().to_list())

In [ ]:
with pl.Config(tbl_rows=20):
        display (train_df.filter (pl.col("country") =="Kenya").group_by (['product', 'store']).agg(pl.col('num_sold').n_unique()))

In [ ]:
with pl.Config(tbl_rows=30) :
        print (train_df.group_by (['product', 'country']).agg(pl.col('num_sold').n_unique()).sort("num_sold"))

In [ ]:
with pl.Config(tbl_rows=20):
        display (train_df.group_by (['store', 'country']).agg(pl.col('num_sold').n_unique()))

In [ ]:
if create_EDA :
    sns.histplot (train_df.filter (pl.col('num_sold').is_not_null()), x= 'num_sold')

In [ ]:
sales_present = train_df.group_by (by ="date").len().sort("by")
display (sales_present)

In [ ]:
cat_features =   ['date', 'num_sold', 'country', 'store',  'product']
for c in cat_features:
        print (f'values for {c}')
        with pl.Config(tbl_rows=50):
            display (train_df.group_by(by = c).agg (pl.col('num_sold').mean(), pl.col('id').len()).sort('id')   )

In [ ]:
low_guys = train_df.filter (pl.col('num_sold') < 45)
print (low_guys.group_by (by = 'country').len())

In [ ]:
dates_train = train_df.with_columns (pl.col("date").str.tail(5).alias ("month_day"))

top_date_group = dates_train.top_k(5000, by = "num_sold").group_by ("month_day").len().sort("len").tail (20)
top_date = top_date_group.get_column ("month_day").to_list()
    
bottom_date_group = dates_train.bottom_k(5000, by = "num_sold").group_by ("month_day").len().sort("len").tail (20)
bottom_date = bottom_date_group.get_column  ("month_day").to_list()
with pl.Config(tbl_rows=20):
    print ("Dates with the most sales")
    display (top_date)
    print ("Dates with the least sales")
    display (bottom_date)
    

In [ ]:
%%time 
if create_EDA :
    display_me = train_df.filter (pl.col('country') == "Finland").group_by(["date", "store"]).agg(pl.col("num_sold").sum())
    sns.lineplot (data = display_me.to_pandas(), x = 'date', y= 'num_sold', hue = "store")

In [ ]:
%%time 
if create_EDA :
    display_me = train_df.filter (pl.col('country') == "Italy").group_by(["date", "product"]).agg(pl.col("num_sold").sum())
    sns.lineplot (data = display_me.to_pandas(), x = 'date', y= 'num_sold', hue = "product")
    print (f"shape of dataset : {display_me.shape}")
    

In [ ]:
%%time 
if create_EDA :
    data_italy = train_df.filter (pl.col('country') == "Italy").with_columns (
        (pl.col("store") + "-" + pl.col("product")).alias ("store_prod"))
    sns.lineplot (data = data_italy.to_pandas(), x = 'date', y= 'num_sold', hue = "store_prod")                                          

In [ ]:
%%time 
if create_EDA :
    sns.lineplot (data = train_df.filter ((pl.col('country') == "Italy") &
                                          (pl.col('product') == "Holographic Goose")).to_pandas(), x = 'date', y= 'num_sold', hue = "store")
    sns.lineplot (data = train_df.filter ((pl.col('country') == "Singapore") &
                                          (pl.col('product') == "Holographic Goose")).to_pandas(), x = 'date', y= 'num_sold', hue = "store")


In [ ]:
print (f"Date start train: {train_df.get_column ('date').min()} , Date end: {train_df.get_column ('date').max()}")

print (f"Date start test: {test_df.get_column ('date').min()} , Date end: {test_df.get_column ('date').max()}")

In [ ]:
%%time 
if create_EDA :
    sns.lineplot (data = train_df.filter (pl.col('country') == "Italy").to_pandas(), x = 'date', y= 'num_sold', hue = "product")

In [ ]:
%%time 
if create_EDA :
    sns.lineplot (data = train_df.filter (pl.col('country') == "Norway").to_pandas(), x = 'date', y= 'num_sold', hue = "store")

In [ ]:
if create_EDA :
    train_df.filter(pl.col("num_sold").is_null()).group_by (["product", "country", "store"]).len()

In [ ]:
sales_per_store_in_country = train_df.group_by (["date", "country", "store"]).agg(pl.col("num_sold").sum()).sort (["date", "country"])
sales_per_store_in_country = sales_per_store_in_country.rename ({"num_sold" : "num_sold_store"})
sales_per_store_in_country = sales_per_store_in_country.with_columns ((pl.col("country") + "_" + 
                                                                        pl.col("date")).alias ("country_date"))

sales_per_country = train_df.group_by (["date", "country"]).agg(pl.col("num_sold").sum()).sort (["date", "country"])
sales_per_country = sales_per_country.rename ({"num_sold" : "num_sold_country"})
sales_per_country = sales_per_country.with_columns ((pl.col("country") + "_" + 
                                                                        pl.col("date")).alias ("country_date"))

ratio_per_store_over_time = sales_per_store_in_country.join (sales_per_country, how = "left", left_on = "country_date", right_on = "country_date",)


ratio_per_store_over_time = ratio_per_store_over_time.drop(["date_right", "country_right"])
ratio_per_store_over_time = ratio_per_store_over_time.with_columns ((pl.col("num_sold_store")/ pl.col("num_sold_country")).alias ("ratio"))
display (ratio_per_store_over_time)

if create_EDA :
    sns.lineplot(data = ratio_per_store_over_time.to_pandas(), x = "date", y = "ratio", hue = "store")

ratio = ratio_per_store_over_time.group_by("store").agg(pl.col("ratio").mean())

print (ratio)


# Feature engineering 

In [ ]:
def add_gdp_per_capita (df : pl.DataFrame, gdp_per_capita : pl.DataFrame ) -> pl.DataFrame :

       result = df.with_columns ((pl.col("country") + "_" + pl.col("date").str.head(4)).alias ("country_year")   )
       lookup = gdp_per_capita.with_columns ((pl.col("Entity") + "_" + pl.col("Year").cast(pl.String)).alias ("country_year")   )
       result = result.join (lookup, how = "left", left_on = "country_year", right_on = "country_year"   )                                      
       return result. drop ([ 'country_year', 'Entity', 'Code', 'Year', '900793-annotations'])

# add_gdp_per_capita (train_df, gdp_per_capita)


In [ ]:
import holidays

years_range = [ n for n in range (2010, 2020)]
ken_holidays = holidays.country_holidays("KE", years = years_range)  
usa_holidays = holidays.country_holidays("US", years = years_range)  
fin_holidays = holidays.country_holidays("FI", years = years_range)  
can_holidays = holidays.country_holidays("CA", years = years_range)  
sin_holidays = holidays.country_holidays("SG", years = years_range)  
ita_holidays = holidays.country_holidays("IT", years = years_range)

def add_holiday (df : pl.DataFrame) -> pl.DataFrame :
    result = df
    result = result.with_columns(pl.lit (False).alias ("is_holiday"))
    result = result.with_columns(pl.when (pl.col("country") == "USA").then (
                      pl.col('p-date').is_in (usa_holidays.keys())).otherwise (
                      pl.col("is_holiday")).alias ('is_holiday'))
    result = result.with_columns(pl.when (pl.col("country") == "Finland").then (
                      pl.col('p-date').is_in (fin_holidays.keys())).otherwise (
                      pl.col("is_holiday")).alias ('is_holiday'))
    result = result.with_columns(pl.when (pl.col("country") == "Canada").then (
                      pl.col('p-date').is_in (can_holidays.keys())).otherwise (
                      pl.col("is_holiday")).alias ('is_holiday'))
    result = result.with_columns(pl.when (pl.col("country") == "Italy").then (
                      pl.col('p-date').is_in (ita_holidays.keys())).otherwise (
                      pl.col("is_holiday")).alias ('is_holiday'))
    result = result.with_columns(pl.when (pl.col("country") == "Kenya").then (
                      pl.col('p-date').is_in (ken_holidays.keys())).otherwise (
                      pl.col("is_holiday")).alias ('is_holiday'))
    result = result.with_columns(pl.when (pl.col("country") == "Singapore").then (
                      pl.col('p-date').is_in (sin_holidays.keys())).otherwise (
                      pl.col("is_holiday")).alias ('is_holiday'))
    return result

In [ ]:
from datetime import date
  

num_to_days = {1 : "Mon", 2 : "Tue", 3 : "Wed", 4 : "Thu", 5  :"Fri", 6 : "Sat", 7 : "Sun"}


def add_features (raw : pl.DataFrame) -> pl.DataFrame :
    result = raw
        
    result = result.with_columns ((pl.col('country') + '_' + 
                                   pl.col('store') + '_' + 
                                   pl.col ('product')).alias ('country_store_product'), 
                          pl.col ('date').str.to_date().alias('p-date')) 
                          
    result = result.with_columns(pl.col ('p-date').dt.weekday ().replace_strict (num_to_days, return_dtype = pl.String ).alias ('weekday'),
                                    pl.col ('p-date').dt.quarter ().alias ('quarter'), 
                                    pl.col ('p-date').dt.year ().alias ('year'), 
                                    pl.col ('p-date').dt.ordinal_day ().alias ('day_of_year'), 
                                    pl.col("date").str.tail(5).alias ("month_day"))
    result = result.with_columns ((pl.col("day_of_year")/365 *2  * np.pi).sin ().alias ("year_sin"), 
                                  (pl.col("day_of_year")/365 *2 * np.pi).cos ().alias ("year_cos"), 
    #                              (pl.col("day_of_year")/365 *4  * np.pi).sin ().alias ("year_sin2x"), 
    #                              (pl.col("day_of_year")/365 *4 * np.pi).cos ().alias ("year_cos2x"),
    #                              ((pl.col("day_of_year") + 90) /365 *2  * np.pi).sin ().alias ("year_sinx_plus90"), 
    #                              ((pl.col("day_of_year") + 90)/365 *2 * np.pi).cos ().alias ("year_cosx_plus90")
                                 )
    result = result.with_columns (pl.col ("month_day").is_in (top_date).alias ("max_date"), 
                                 pl.col ("month_day").is_in (bottom_date).alias ("min_date"))
    
    result = add_holiday (result)                             
    
    result = result.with_columns(pl.col ('weekday').is_in ([ 'Fri', 'Sat', 'Sun']).alias ('is_weekend'))
    result = add_gdp_per_capita (result, gdp_per_capita)
    
    if "num_sold" in raw.columns :
        result = result.with_columns (pl.col ("num_sold").log().alias ("num_sold_log"))
        
    
        
    return result

In [ ]:
train_added_features = add_features (train_df)

train_added_features


In [ ]:

if create_EDA :
    sns.lineplot (data = train_added_features.filter (pl.col("country_store_product") == "Canada_Discount Stickers_Kaggle").to_pandas(), x = "date", y = "num_sold_log")

In [ ]:
%%time 
if create_EDA :

    sns.histplot (data = train_added_features.to_pandas(), x = "num_sold_log")
    

In [ ]:
%%time 
if create_EDA :

    sns.histplot (data = train_added_features.to_pandas(), x = "num_sold")

In [ ]:
%%time 
if create_EDA :
    sns.histplot (data = train_added_features.filter (pl.col("country") == "Kenya").to_pandas(), x = "num_sold")

In [ ]:
%%time 
if create_EDA :
    sns.histplot (data = train_added_features.filter (pl.col("country") == "Finland").to_pandas(), x = "num_sold")

# Bootstrap : predict missing value in training data set 

In [ ]:
train_no_null=train_added_features.filter (pl.col("num_sold").is_not_null())
train_null=train_added_features.filter (pl.col("num_sold").is_null())

In [ ]:
%%time 
from catboost import CatBoostRegressor, Pool

cat_params = {'iterations': 1000, # Number of boosting iterations (trees)
              'learning_rate': 0.27, # Step size shrinkage for preventing overfitting
              'depth': 8, # Maximum depth of each tree
              'l2_leaf_reg': 0.005, # L2 regularization on leaf values
              'border_count': 250, # Number of splits to consider for features
              'subsample': 0.64, # Fraction of data used for each tree (bagging)
              'random_strength': 5 # Controls randomness in feature splits
             }

X_data = train_no_null.drop(["num_sold", "num_sold_log","p-date"])
print (X_data.schema)

X_train = Pool (X_data.to_pandas(), train_no_null.get_column("num_sold").to_numpy(), 
                cat_features=[1,2,3,4,5,6,7,10 ])

y_train = train_no_null.get_column("num_sold")
X_boot  = train_null.drop("num_sold")
cat_model = CatBoostRegressor(**cat_params, verbose=False)
cat_model.fit(X_train)

In [ ]:
%%time 
X_data = train_null.drop(["num_sold", "num_sold_log","p-date"])
X_boot_pool = Pool (X_data.to_pandas(),  
                cat_features=[1,2,3,4,5,6,7,10 ])


cat_model_predict = pl.max_horizontal (pl.Series ("num_sold", cat_model.predict(X_boot_pool)).round(0), 0)

In [ ]:
print (cat_model_predict)
X_boot  = train_null.drop("num_sold")
X_boot_result = train_null.with_columns (cat_model_predict.alias ("num_sold"))

complete_time_series_data = pl.concat ([train_no_null, X_boot_result], how = "vertical_relaxed").drop("num_sold_log")
                            
complete_time_series_data.select ("num_sold").describe ()                            

In [ ]:
# holographic_goose_factor = 197/(651+1018+556+1232)
# holographic_goose_factor
# print (f"{holographic_goose_factor = }")

# stickers_for_less_factor = 840 / (840 +973 + 427)
# premium_sticker_market_factor = 973 / (840 +973 + 427)
# discount_stickers_factor = 427 / (840 +973 + 427)

# print (f"{stickers_for_less_factor = }, {premium_sticker_market_factor = } , {discount_stickers_factor = }")

In [ ]:
# print (train_df.filter ((pl.col("country") == "Kenya") & 
#                         (pl.col("product") == "Holographic Goose") & 
#                         # (pl.col("num_sold").is_null())).group_by ("store").len())
# only_Kenya = train_df.filter (pl.col("country") == "Kenya")

# train_Kenya_per_date = only_Kenya.group_by ("date").agg (pl.col("num_sold").sum()).sort ("date")

# train_Kenya_per_date = train_Kenya_per_date.with_columns (
#                             pl.max_horizontal (1, (pl.col ("num_sold") * 0.05698582586057275 * 0.375 - 2)).round(0).alias ("Stickers for Less"), 
#                             pl.max_horizontal (1, (pl.col ("num_sold") * 0.05698582586057275 * 0.434375 - 2)).round(0).alias ("Premium Sticker Mart"), 
#                             pl.max_horizontal (1, (pl.col ("num_sold") * 0.05698582586057275 * 0.190625 - 2)).round(0).alias ("Discount Stickers"))

# train_Kenya_per_date = train_Kenya_per_date.drop("num_sold")





In [ ]:
# import polars.selectors as cs
# train_Kenya_unpivot = train_Kenya_per_date.unpivot (cs.numeric(), index = "date")
# train_Kenya_unpivot = train_Kenya_unpivot.rename ({"value" : "num_sold_calc", 
#                                    "variable" : "store"})
# train_Kenya_unpivot = train_Kenya_unpivot.with_columns (pl.lit("Holographic Goose").alias ("product"),
#                                         pl.lit("Kenya").alias ("country"))
# print (train_Kenya_unpivot.head (3))

# compare_Kenya = train_Kenya_unpivot.join(only_Kenya, how = "left", on =["country", "store","date","product"])

# compare_Kenya = compare_Kenya.with_columns ((pl.col("num_sold") - pl.col("num_sold_calc")).alias ("delta"))

In [ ]:
# import seaborn as sns


# sns.lineplot (data = compare_Kenya.to_pandas (), x = "date", y = "num_sold", hue = "store", palette='Oranges')
# sns.lineplot (data = compare_Kenya.to_pandas (), x = "date", y = "num_sold_calc", hue = "store", palette='Greens')


# train_Kenya = compare_Kenya.with_columns (pl.when(pl.col("num_sold").is_null()).then (
#                                            pl.col("num_sold_calc")).otherwise(
#                                            pl.col("num_sold")).alias ("num_sold")    
#                                            )


# train_Kenya = train_Kenya.drop ( ["delta","num_sold_calc"])

# print (train_Kenya.select (cs.numeric()).describe ())

In [ ]:
# print (train_df.filter ((pl.col("country") == "Canada") & 
#                         (pl.col("product") == "Holographic Goose") & 
#                         (pl.col("num_sold").is_null())).group_by ("store").len())
# # only_Canada = tain_df.filter (pl.col("country") == "Canada")

# train_Canada_per_date = only_Canada.group_by ("date").agg (pl.col("num_sold").sum()).sort ("date")

# train_Canada_per_date = train_Canada_per_date.with_columns (
#                             pl.max_horizontal (1, (pl.col ("num_sold") * 0.05698582586057275 * 0.375 - 34)).round(0).alias ("Stickers for Less"), 
#                             pl.max_horizontal (1, (pl.col ("num_sold") * 0.05698582586057275 * 0.434375 - 34)).round(0).alias ("Premium Sticker Mart"), 
#                             pl.max_horizontal (1, (pl.col ("num_sold") * 0.05698582586057275 * 0.190625 - 34)).round(0).alias ("Discount Stickers"))

# train_Canada_per_date = train_Canada_per_date.drop("num_sold")



In [ ]:
# import polars.selectors as cs
# train_Canada_unpivot = train_Canada_per_date.unpivot (cs.numeric(), index = "date")
# train_Canada_unpivot = train_Canada_unpivot.rename ({"value" : "num_sold_calc", 
#                                    "variable" : "store"})
# train_Canada_unpivot = train_Canada_unpivot.with_columns (pl.lit("Holographic Goose").alias ("product"),
#                                         pl.lit("Canada").alias ("country"))
# print (train_Canada_unpivot.head (3))

# compare_Canada = train_Canada_unpivot.join(only_Canada, how = "left", on =["country", "store","date","product"])

# compare_Canada = compare_Canada.with_columns ((pl.col("num_sold") - pl.col("num_sold_calc")).alias ("delta"))

In [ ]:
# import seaborn as sns

# sns.lineplot (data = compare_Kenya.to_pandas (), x = "date", y = "num_sold", hue = "store", palette='Oranges')
# sns.lineplot (data = compare_Kenya.to_pandas (), x = "date", y = "num_sold_calc", hue = "store", palette='Greens')

# print (compare_Canada.filter (pl.col("num_sold").is_not_null()).head ())                                                                
# print (compare_Canada.select (cs.numeric()).describe ())

# train_Canada = compare_Canada.with_columns (pl.when(pl.col("num_sold").is_null()).then (
#                                            pl.col("num_sold_calc")).otherwise(
#                                            pl.col("num_sold")).alias ("num_sold")    
#                                            )


# train_Canada = train_Canada.drop ( ["delta","num_sold_calc"])

# print (train_Canada.select (cs.numeric()).describe ())

In [ ]:
# estimates = pl.concat ([train_Kenya, train_Canada], how = "vertical")
# estimates = estimates.with_columns(pl.col("num_sold").log().alias ("num_sold_log"))
# print (estimates)
# estimated_target = train_null.drop(["num_sold", "num_sold_log"]).join(estimates, on =["date", "country", "store", "product"], how = "left")

# estimated_target = estimated_target.drop ("id_right")
# print  (estimated_target.head())

In [ ]:
# print (f"{train_no_null.columns = }")
# print (f"{estimated_target.columns = }")
# column_order = estimated_target.columns
# complete_time_series_data = pl.concat ([train_no_null.select (column_order), estimated_target], how = "vertical_relaxed")
# print (complete_time_series_data.select (['num_sold']) .describe() )



In [ ]:
complete_time_series_data.filter (pl.col("num_sold").is_null()).group_by(["country", "store", "product"]).len()

# Install Autogluon and related packages

In [ ]:
!pip install ray==2.10.0

In [ ]:
!pip install autogluon.timeseries --no-cache-dir -q
!pip install -U ipywidgets

In [ ]:
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

from autogluon.tabular import TabularPredictor

from autogluon.common import space

print ('done')

# Time Series training with Auto Gluon for Time Series 

In [ ]:
def create_static_features (df : pl.DataFrame) -> pl.DataFrame :
    all_ids = df.get_column ('country_store_product').unique()
    result = pl.DataFrame(all_ids, schema = {"country_store_product" : pl.String}) 
    result = result.with_columns (pl.col('country_store_product').str.split ("_").list.get (0).alias ("country"), 
                                    pl.col('country_store_product').str.split ("_").list.get(1).alias ("store"),
                                    pl.col('country_store_product').str.split ("_").list.get (2).alias ("product"))
    return result
    

In [ ]:
%%time

known_features_for_train = add_features (complete_time_series_data).drop('GDP per capita_right')
print (known_features_for_train.columns)

known_features_for_train = known_features_for_train.rename ({"country_store_product" : "item_id", 
                                                             "date" : "timestamp"})
known_features_for_train_pandas = known_features_for_train.to_pandas()

#known_features_for_train_pandas.set_index('country_store_product', inplace=True)
print (complete_time_series_data.columns)

prep = complete_time_series_data.drop(["id", "country", "store" ,"product",
                                       "p-date"])   

train_data = TimeSeriesDataFrame.from_data_frame(
            prep.to_pandas(),
            id_column="country_store_product",
            timestamp_column="date")

country_store_product_features = create_static_features (prep)
print (" static features for Auto Gluon")
print (country_store_product_features)
print (f"country_store_product_features.shape = {country_store_product_features.shape}")

print (country_store_product_features.get_column ("country").unique())
print (country_store_product_features.get_column ("store").unique())
print (country_store_product_features.get_column ("product").unique())

country_store_product_features_pandas = country_store_product_features.to_pandas() 
country_store_product_features_pandas.set_index('country_store_product', inplace=True)
train_data.static_features = country_store_product_features_pandas

print (train_data.head())

In [ ]:
%%time


known_features = ["weekday", "quarter", "year", "day_of_year", "year_sin", "year_cos",  
                 # "year_sin2x", "year_cos2x", "year_sinx_plus90", "year_cosx_plus90",
                  "is_holiday", "month_day", "is_weekend", "min_date", "max_date", "GDP per capita"]

print (f"train_data.columns = {train_data.columns}")
print (f"known_features = {known_features}")

predictor = TimeSeriesPredictor(path = f'/kaggle/working/autogluon_time_series',
                            label='num_sold', 
                            freq="D",  
                            eval_metric =  'MAPE', 
                            known_covariates_names= known_features,
                            prediction_length = 1095,   # matches the time window from the test data 
                            learner_kwargs = {'ignored_columns' : [
                                   'id', 
                                  'my_weight'
                                    ]})



predictor.fit(train_data= train_data, 
                    presets= 'best_quality',
                        
# best_quality,  medium_quality,                      
#                    time_limit = 20000,
                    refit_full = True,
                    num_val_windows = 1,
                        # num_gpus=1, 
                        # dynamic_stacking=False, num_stack_levels=1
#                        hyperparameters=hyperparameters,
#                        hyperparameter_tune_kwargs=hyperparameter_tune_kwargs,
                        )

predictor.refit_full()

# How did we do?

In [ ]:
predictor.leaderboard()

# Prediction Time Series for Submission 

In [ ]:
%%time

known_features_for_test = add_features (test_df)

requested_forecast = known_features_for_test.get_column ("p-date").max() - known_features_for_test.get_column ("p-date").min()

print (f"requested froecast = {requested_forecast}")
known_features_for_test_ts = TimeSeriesDataFrame.from_data_frame(
            known_features_for_test.to_pandas(),
            id_column="country_store_product",
            timestamp_column="date")
# known_features_for_test_to_pandas.set_index('country_store_product', inplace=True)

predictions = predictor.predict(train_data, known_covariates = known_features_for_test_ts)

In [ ]:
# creating a Dataframe that can be used for a join with the original Test data using the unique key "country_store_product_timestamp"
predictions_df = pl.DataFrame (predictions.reset_index())
predictions_df = predictions_df.with_columns (pl.col("timestamp").cast (pl.String).str.head(10))

predictions_df = predictions_df.with_columns ((pl.col('item_id') + "_" +  pl.col('timestamp')).alias ("country_store_product_timestamp"), 
                                               pl.col('mean').alias ('num_sold')) 

#predictions_df = predictions_df.with_columns ((pl.col('item_id') + "_" +  pl.col('timestamp')).alias ("country_store_product_timestamp"), 
#                                               pl.col('mean').exp().alias ('num_sold')) 

In [ ]:
predictions_df = predictions_df.drop (["timestamp",  "item_id", "mean", "0.1","0.2", "0.3", "0.4", "0.5", "0.6", "0.7", "0.8", "0.9", ])
print (predictions_df.columns)
print (predictions_df.select ("num_sold").describe)

# submission

In [ ]:
# join the prediction time series with the test data set 

test_lookup_df = test_df.with_columns ((pl.col("country") + "_" +
                                        pl.col("store") + "_" + 
                                        pl.col("product") + "_" + 
                                        pl.col ("date")).alias ("country_store_product_timestamp"))
print (f"test_lookup_df.columns = {test_lookup_df.columns}")
print (f"predictions_df.columns = {predictions_df.columns}")
                                               
test_results = test_lookup_df.join (predictions_df,
                                 how = 'left', 
                                 left_on = "country_store_product_timestamp",
                                 right_on = "country_store_product_timestamp")

test_results

In [ ]:

submission = test_results.select (["id", "num_sold"])

submission = submission.with_columns (pl.max_horizontal( pl.col("num_sold").round(0), 0).alias ("num_sold"))

# submission = submission.drop ("num_log")


In [ ]:
print (submission.describe())
print (submission)
submission.write_csv(f'submission.csv')


# How did it go?

In [ ]:

predictor.plot(train_data, predictions, item_ids = ["Canada_Premium Sticker Mart_Holographic Goose",
                                                    "Canada_Premium Sticker Mart_Kaggle",
                                                    "Canada_Premium Sticker Mart_Kaggle Tiers",
                                                    "Canada_Premium Sticker Mart_Kerneler",
                                                    "Canada_Premium Sticker Mart_Kerneler Dark Mode",
                                                    "Canada_Stickers for Less_Holographic Goose",
                                                    "Canada_Stickers for Less_Kaggle",
                                                    "Canada_Stickers for Less_Kaggle Tiers",
                                                    "Canada_Stickers for Less_Kerneler",
                                                    "Canada_Stickers for Less_Kerneler Dark Mode",
                                                    "Canada_Discount Stickers_Holographic Goose",
                                                    "Canada_Discount Stickers_Kaggle",
                                                    "Canada_Discount Stickers_Kaggle Tiers",
                                                    "Canada_Discount Stickers_Kerneler",
                                                    "Canada_Discount Stickers_Kerneler Dark Mode",
                                                   ])

In [ ]:
predictor.plot(train_data, predictions, item_ids = ["Italy_Premium Sticker Mart_Holographic Goose",
                                                    "Italy_Premium Sticker Mart_Kaggle",
                                                    "Italy_Premium Sticker Mart_Kaggle Tiers",
                                                    "Italy_Premium Sticker Mart_Kerneler",
                                                    "Italy_Premium Sticker Mart_Kerneler Dark Mode",
                                                    "Italy_Stickers for Less_Holographic Goose",
                                                    "Italy_Stickers for Less_Kaggle",
                                                    "Italy_Stickers for Less_Kaggle Tiers",
                                                    "Italy_Stickers for Less_Kerneler",
                                                    "Italy_Stickers for Less_Kerneler Dark Mode",
                                                    "Italy_Discount Stickers_Holographic Goose",
                                                    "Italy_Discount Stickers_Kaggle",
                                                    "Italy_Discount Stickers_Kaggle Tiers",
                                                    "Italy_Discount Stickers_Kerneler",
                                                    "Italy_Discount Stickers_Kerneler Dark Mode",
                                                   ])

In [ ]:
predictor.plot(train_data, predictions, item_ids = ["Kenya_Premium Sticker Mart_Holographic Goose",
                                                    "Kenya_Premium Sticker Mart_Kaggle",
                                                    "Kenya_Premium Sticker Mart_Kaggle Tiers",
                                                    "Kenya_Premium Sticker Mart_Kerneler",
                                                    "Kenya_Premium Sticker Mart_Kerneler Dark Mode",
                                                    "Kenya_Stickers for Less_Holographic Goose",
                                                    "Kenya_Stickers for Less_Kaggle",
                                                    "Kenya_Stickers for Less_Kaggle Tiers",
                                                    "Kenya_Stickers for Less_Kerneler",
                                                    "Kenya_Stickers for Less_Kerneler Dark Mode",
                                                    "Kenya_Discount Stickers_Holographic Goose",
                                                    "Kenya_Discount Stickers_Kaggle",
                                                    "Kenya_Discount Stickers_Kaggle Tiers",
                                                    "Kenya_Discount Stickers_Kerneler",
                                                    "Kenya_Discount Stickers_Kerneler Dark Mode",
                                                   ])

# Cleanup

In [ ]:
%%time 
import zipfile, os


def zip_files_in_directory(directory, zip_name):
    num_files_deleted = 0 
    with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(directory):
            for file in files:
                file_path = os.path.join(root, file)
                zipf.write(file_path, os.path.relpath(file_path, directory))
                os.remove(file_path)
                num_files_deleted += 1
    return num_files_deleted  
# Example usage
directory = '/kaggle/working/autogluon_time_series'
zip_name = 'autogluon_time_series.zip'
n = zip_files_in_directory(directory, zip_name)
print(f"deleted {n } files in {directory}")

directory = '/kaggle/working/Autogluon_bootstrap'
zip_name = 'Autogluon3.zip'
m = zip_files_in_directory(directory, zip_name)

print(f"deleted { m} files in {directory}")